<a href="https://colab.research.google.com/github/timothy-shiwprasad/COMP6905_assigment_2/blob/main/Assignment_2_Question_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question 3 : Plannning a Pokemon Rescue Expedition


A)

Goal: Maximize total rescue points

| Pokémon              | Viridian Forest | Mt. Moon | Seafoam Islands | Deployment cost |
|----------------------|----------------:|---------:|----------------:|----------------:|
| Charizard            | 11              | 13       | 4               | 4               |
| Blastoise             | 8               | 7        | 12              | 4               |
| Pikachu               | 10              | 8        | 6               | 3               |
| Raichu                | 9               | 10       | 7               | 3               |
| Pidgeot               | 10              | 9        | 7               | 3               |
| Machamp               | 7               | 12       | 6               | 5               |
| Lapras                | 5               | 6        | 14              | 4               |
| **Outpost opening cost** | 6            | 8        | 9               |                 |

### Rescue Points and Cost.

Rules:
- At least 2 outposts should be open.
- Each outpost should have 2 to 3 pokemon assigned.
- A closed outpost should have 0 pokemon assigned.
- At most one of Pikachu and Raichu may be deployed anywhere.
- If Charizard is assigned to an outpost , Blastoise or Lapras must also be assigned to that outpost
- Seafoam Island outpost may be open if lapras is assigned there.
- Fire Safety Kit is installed if Charizard is assinged to that outpost.
- Budget is 50

Sets:

### Outposts

$ J = {Viridian Forest,Mt.Moon,Seafoam Islands} $

### Pokemons
 $ I = {Charizard,Blastoise,Pikachu,Raichu,Pidgeot,Machamp,lapras} $

### Parameters

$p_{ij}$ = Rescue  Points of Pokemon i at outpost j

$d_i$ = deployment cost of pokemon i

$o_j$ = Opening Cost of outpost j

Kit Cost = 2

Budget B = 50


# Binary Variables


### Assigment Variable
$$
x_{ij} =
\begin{cases}
1, & \text{if pokemon } i \text{ is assigned to outpost} j \\
0, & \text{otherwise}
\end{cases}
$$

### Outpost Opening Variable

$$
y_{j} =
\begin{cases}
1, & \text{if outpost } j \text { is open} \\
0, & \text{otherwise}
\end{cases}
$$

### Fire Safety Kit
$$
f_{j} =
\begin{cases}
1, & \text{if a fire safety kit is installed at outpost} j \\
0, & \text{otherwise}
\end{cases}
$$

## Objective Function

$$ max \sum_i \sum_j p_{ij} x_{ij}$$

## Constraints

- Each Pokemon must work at most 1 outpost

$$ \sum_j x_{ij} <= 1 $$

- Open Outpost must have 2 to 3 Pokemon

$$ 2y_j <= \sum_i x_{ij} <= 3y_j $$

- At Least 2 outposts open

$$ \sum_{j} y_j >= 2 $$

- At most one of Pikachu and Raichu

$$ \sum_j (x_{Pikachuj)}+ x_{Raichuj}) <= 1 $$

- Charizard must be accompanied by Blastoise or Lapras at a location

$$ x_{charizardj} <= x_{Blastoisej} + x_{Laprasj} $$

- Seafoam is open only if Lapras is there

$$y_{Seafoam} <= x_{lapras,Seafoam}$$

- Install Fire safety kit if and only if Charizard is there.

$$ f_j = x_{Charizard,j} $$

## Justification of Fire Safety Kit constraint

There are two implications , if charizard is at J , then there is a Fire safety Kit at J. The second is if there is a Kit at J then Charizard is assigned at J. Therefore if Charizard is at J there is a Kit at J and if a Kit is at J there is a Charizard there. Since we are dealing with Binary variables if $x_{charizard,j}$ is 1 or 0 then f_j will be 1 or 0 respectively.

- Budget Constraint

$$ \sum_{j} o_{j} y_j + \sum_i d_i \sum_j x_{ij} + 2\sum_j k_j <= 50 $$


## Final Objective Function

$$ max \sum_i \sum_j p_{ij} x_{ij}$$

subject to:

$$ \sum_j x_{ij} <= 1 $$

$$ 2y_j <= \sum_i x_{ij} <= 3y_j $$

$$ \sum_{j} y_j >= 2 $$

$$ \sum_j (x_{Pikachuj)}+ x_{Raichuj}) <= 1 $$

$$ x_{charizardj} <= x_{Blastoisej} + x_{Laprasj} $$

$$y_{Seafoam} <= x_{lapras,Seafoam}$$

$$ f_j = x_{Charizard,j} $$

$$ \sum_{j} o_{j} y_j + \sum_i d_i \sum_j x_{ij} + 2\sum_j k_j <= 50 $$

$$ x_{ij} , y_j , k_j \in {0,1} $$


In [ ]:
# Using an older pulp package than what is avaliable in the Notebook.
# Please run here to install.
# If package version is 4.x.x still , restart the notebook and run the cell again.
# Code will fail without this package.
!pip install -q pulp==2.9.0
import pulp
print(pulp.__version__)

2.9.0


In [ ]:
# Question 3 B
# reused for part C
import pulp


## Prepare the Data.
pokemon  = ["Charizard","Blastoise","Pikachu","Raichu","Pidgeot","Machamp","Lapras"]
outposts = ["Viridian","MtMoon","Seafoam"]

## Rescue Points
points = {
    "Charizard":[11,13, 4], "Blastoise":[ 8, 7,12], "Pikachu":[10, 8, 6],
    "Raichu":   [ 9,10, 7], "Pidgeot":  [10, 9, 7], "Machamp":[ 7,12, 6],
    "Lapras":   [ 5, 6,14]}
p = {(i,j): points[i][n] for i in pokemon for n,j in enumerate(outposts)}

## Deployment Cost
deploy    = {"Charizard":4,"Blastoise":4,"Pikachu":3,"Raichu":3,
             "Pidgeot":3,"Machamp":5,"Lapras":4}

## Cost to Open Center
open_cost = {"Viridian":6,"MtMoon":8,"Seafoam":9}

KIT = 2
BUDGET = 50

In [ ]:
# reused for part C
def solve_rescue(backup_rule=True):
    m = pulp.LpProblem("Pokemon_Rescus_Expedition", pulp.LpMaximize)
    x = {i:{j: pulp.LpVariable(f"x_{i}_{j}", cat="Binary") for j in outposts} for i in pokemon}
    y = {j: pulp.LpVariable(f"y_{j}", cat="Binary") for j in outposts}
    f = {j: pulp.LpVariable(f"k_{j}", cat="Binary") for j in outposts}

    # objective
    m += pulp.lpSum(p[i,j]*x[i][j] for i in pokemon for j in outposts)

     # C1 Each Pokemon to One outpost
    for i in pokemon:
        m += pulp.lpSum(x[i][j] for j in outposts) <= 1

    # C2 2 to 3 Pokemon per outposts
    for j in outposts:
        m += pulp.lpSum(x[i][j] for i in pokemon) >= 2*y[j]
        m += pulp.lpSum(x[i][j] for i in pokemon) <= 3*y[j]

    # C3 At Least 2 outposts open
    m += pulp.lpSum(y.values()) >= 2
    # C4 Either Pikachu or Raichu to be used for the entire expedetion
    m += pulp.lpSum(x[i][j] for i in ["Pikachu","Raichu"] for j in outposts) <= 1

    # C5 Charizard must be accomodated by Blastoise or Lapras
    if backup_rule:
        for j in outposts:
            m += x["Charizard"][j] <= x["Blastoise"][j] + x["Lapras"][j]
    # C6 Seafoam is only open if lapras was deployed there
    m += y["Seafoam"] <= x["Lapras"]["Seafoam"]

    # C7 Fire Kit installed if Charizard is there.
    for j in outposts:
        m += f[j] == x["Charizard"][j]

    # C8
    cost = (pulp.lpSum(open_cost[j]*y[j] for j in outposts)
          + pulp.lpSum(deploy[i]*x[i][j] for i in pokemon for j in outposts)
          + KIT*pulp.lpSum(f.values()))
    m += cost <= BUDGET


    m.solve(pulp.PULP_CBC_CMD(msg=0))
    print(pulp.LpStatus[m.status], "| points =", pulp.value(m.objective), "| cost =", pulp.value(cost))
    for j in outposts:
        team = [i for i in pokemon if x[i][j].value() > 0.5]
        print(f"{j:9s} | {'OPEN' if y[j].value()>0.5 else 'closed'} | {team} | kit: {f[j].value()>0.5}")

In [ ]:
# Question 3 B
solve_rescue(True)

Optimal | points = 62.0 | cost = 48.0
Viridian  | OPEN | ['Charizard', 'Blastoise'] | kit: True
MtMoon    | OPEN | ['Raichu', 'Machamp'] | kit: False
Seafoam   | OPEN | ['Pidgeot', 'Lapras'] | kit: False


# Question 3 B

## Model Output

Maximum Resuce Points: 62

Total Cost :48

Outposts and assigment:

| Outpost  | Status | Team                          | Kit   |
|----------|--------|-------------------------------|-------|
| Viridian | OPEN   | Charizard, Blastoise          | True  |
| MtMoon  | OPEN    | Raichu, Machamp               | False |
| Seafoam  | OPEN   | Pidgeot,Lapras          | False |



In [ ]:
# Question 3 C


# We remove the Charizard Dependency on Blastoise and Lapras
solve_rescue(False)

Optimal | points = 71.0 | cost = 48.0
Viridian  | OPEN | ['Pikachu', 'Pidgeot'] | kit: False
MtMoon    | OPEN | ['Charizard', 'Machamp'] | kit: True
Seafoam   | OPEN | ['Blastoise', 'Lapras'] | kit: False


# Question 3 C

## Model Output

Maximum Resuce Points: 71

Total Cost : 48

Outposts and assigment:

| Outpost  | Status | Team                 | Kit   |
|----------|--------|----------------------|-------|
| Viridian | OPEN   | Pikachu, Pidgeot     | False |
| MtMoon   | OPEN   | Charizard, Machamp   | True  |
| Seafoam  | OPEN   | Blastoise, Lapras    | False |

# Comparison with Plan from B

The remval of the Charizard dependency constraint has significantly increased the Resuce Points from 62 to 71. The Cost remained the same at 48. The team assigments are more aligned with maximizing the rescue points. Escpecially for Seafoam for which Blastoise and Lapras are assigned and have the highest rescuse points for this location.
